# Sistema RSA para firmas digitales

En este notebook, mostraremos el proceso para generar las llaves pública y privada, así como el proceso para generar una firma digital y su verificación correspondiente usando el sistema RSA. Empezaremos por cargar las funciones que necesitamos:

In [8]:
import sys
import importlib.util

if 'google.colab' in sys.modules:
    runtime = "Google Colab"
    if importlib.util.find_spec("cryptocalc") is None:
        print("   Installing MA2006B from GitHub\n")
        !pip install git+https://github.com/Krul-dev/MA2006B.git
else:
    runtime = "Local environment"

import cryptocalc

print(f"\n========= Notebook execution context =========")
print(f"              Runtime: {runtime}")
print(f"       Python version: {sys.version.split()[0]}")
print(f"   CryptoCalc version: {cryptocalc.__version__}\n")

from cryptocalc import (
    rsa_key_generation,
    rsa_encryption,
    rsa_decryption,
    sha256_of_sentence,
)


========= Notebook execution context =========
              Runtime: Local environment
       Python version: 3.14.3
   CryptoCalc version: 0.1.0



## Protocolo RSA para la generación de llaves

Una vez que ya hemos cargado las librerías necesarias, empezamos por generar nuestras llaves *pública* y *privada.*

In [9]:
(public_key, private_key) = rsa_key_generation(1000)
d = private_key
e = public_key[0]
n = public_key[1]

print(f"Llave privada (d): {d}\n")
print(f"Llave pública (e): {e}\n")
print(f"Módulo para las llaves (n): {n}\n")

Llave privada (d): 25241009464983598788618922792297417039118998662380334475898299095746999690978779832877530381486791757453018011617541683859951594030758987734845542216598630126530992077849563080082416201194627331650753164789473985497073721088405499218876508418757128982940531130033678752185433121178892026177724579951973544653239176764755251794678819183260375084445190294228481989093789117599997801801643005356021912320928825757747971309439482526186038416062000516732142171480837525867406848807230540545700772421757237798570987188651476057702222644245930018983468703460687038984591011199620815439707910980323587665072485

Llave pública (e): 5350605399856052905462595795326361628184100106739142976414139442580839079221670755933187697206412722648953025664932973894967047011633068252119213597017936363454659680729841585074150438718748312088278281449292990358365787712497766787153846044867828716839867223922546661135593528847448616409152368525289412779166870843085409452289459503910663540843437969627517

Recordemos que el valor $d$ de la llave privada se debe de mantener en secreto. En cambio los valores de $e$ y $n$ corresponden a la llave pública y son conocidos por todos los agentes involucrados, incluída Eva.

## Protocolo RSA para la firma de mensajes

Para ilustrar el algoritmo de generación de firmas digitales, supongamos que Alicia desea firmar el siguiente mensaje llano $m$:

In [10]:
m = "Hello World!"
h = sha256_of_sentence(m)

print(f"Mensaje llano (m): {m}\n")
print(f"Hash del mensaje llano (h): {h}\n")

Mensaje llano (m): Hello World!

Hash del mensaje llano (h): 57676413081093003148005107550719583540116985236696423860923466490497932824681



Para generar la firma digital $s$, Alicia debe de utilizar el mismo algoritmo que utiliza para descifrar mensajes. En otras palabras, Alicia debe calcular
$$
s = h^{d} \mod n
$$
y ya con esta información puede generar el *mensaje firmado*
$$
\text{mensaje\_firmado} = (m, s, n)
$$

In [ ]:
s = rsa_decryption(public_key, private_key, h)
signed_message = (m, s, n)

print(f"El mensaje firmado es: {signed_message}\n")

Para verificar la firma, Beto aplica ahora el mismo algoritmo que utilizaría para encriptar mensajes. De manera más precisa, Beto calcula el Hash
$$
h = \operatorname{Hash}(m)
$$
y calcula también
$$
\tilde{h} = s^{e} \mod n
$$
Si estos dos números son iguales, entonces la firma es válida.

In [ ]:
h = sha256_of_sentence(signed_message[0])
s = signed_message[1]
h_tilde = rsa_encryption(public_key, s)

if h == h_tilde:
    verificacion_firma = "la firma es válida"
else:
    verificacion_firma = "la firma es inválida"

print(f"El valor de h es: {h}\n")
print(f"El valor de h̃ es: {h_tilde}\n")
print("Por lo tanto,", verificacion_firma)